# Resilient Plant · 02 · Ratio Analysis & Volatility Envelope

**Task 2 (reframed).** Cross-sectional ratio analysis across all 1,200+ company-years. We compute year-over-year sector medians, percentile bands (25/50/75/90), and 5-year volatility for Liquidity, Solvency, and Profitability ratios.

**Output:** the empirical envelope the project must operate within — drawn from real data, not extrapolated forecasts.

## Setup — auto-resolving paths

Run this cell first.

In [ ]:
# Auto-resolving path setup
# Folder layout expected:
#   Module 01/
#     ├── Track 03/
#     │   ├── dataset/                   (5 CSVs: 2014..2018_Financial_Data.csv)
#     │   └── resilient_plant/
#     │       ├── data/
#     │       ├── outputs/
#     │       ├── figures/
#     │       └── (notebooks live here, or in a notebooks/ subfolder)

from pathlib import Path

def find_project_root():
    p = Path.cwd().resolve()
    for parent in [p] + list(p.parents):
        if parent.name == "resilient_plant":
            return parent
        if (parent / "outputs").exists() and (parent / "data").exists() and (parent / "figures").exists():
            return parent
    return Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

PROJECT_ROOT = find_project_root()
DATASET_DIR  = PROJECT_ROOT.parent / "dataset"
DATA_DIR     = PROJECT_ROOT / "data"
OUTPUTS_DIR  = PROJECT_ROOT / "outputs"
FIGURES_DIR  = PROJECT_ROOT / "figures"

for d in [DATA_DIR, OUTPUTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Dataset dir  : {DATASET_DIR}")
print(f"Data dir     : {DATA_DIR}")
print(f"Outputs dir  : {OUTPUTS_DIR}")
print(f"Figures dir  : {FIGURES_DIR}")

# Locate the 5 yearly CSVs — accept dataset/ or data/
YEARS = [2014, 2015, 2016, 2017, 2018]
csv_paths = {}
for y in YEARS:
    name = f"{y}_Financial_Data.csv"
    for folder in [DATASET_DIR, DATA_DIR]:
        if (folder / name).exists():
            csv_paths[y] = folder / name
            break
assert len(csv_paths) == 5, (
    f"Expected 5 CSVs (2014..2018_Financial_Data.csv) in {DATASET_DIR} or {DATA_DIR}; "
    f"found {sorted(csv_paths.keys())}"
)
print(f"Found 5 yearly CSVs : {sorted(csv_paths.keys())}")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

clean = pd.read_csv(DATA_DIR / "metals_cleaned.csv")
print(f"Loaded {len(clean):,} metals company-years")


## Liquidity ratios — sector envelope

In [ ]:
liquidity_ratios = ["currentRatio", "quickRatio", "cashRatio"]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
percentiles = [0.10, 0.25, 0.50, 0.75, 0.90]
band_colors = ["#FCE4D6", "#F4B084", "#1F3864", "#F4B084", "#FCE4D6"]
band_alphas = [0.25, 0.4, 1.0, 0.4, 0.25]

for ax, ratio in zip(axes, liquidity_ratios):
    pct = clean.groupby("year")[ratio].quantile(percentiles).unstack()
    # Fill bands
    ax.fill_between(pct.index, pct[0.10], pct[0.90],
                    alpha=0.20, color="#5B9BD5", label="10th-90th pctile")
    ax.fill_between(pct.index, pct[0.25], pct[0.75],
                    alpha=0.40, color="#1F77B4", label="25th-75th pctile")
    ax.plot(pct.index, pct[0.50], marker="o", linewidth=2.5,
            color="#1F3864", label="Median")
    ax.set_title(f"{ratio}", fontweight="bold")
    ax.set_xlabel("Year"); ax.set_ylabel("Ratio")
    ax.set_xticks(pct.index)
    ax.legend(loc="best", fontsize=8)

plt.suptitle("LIQUIDITY · Sector Volatility Envelope (2014-2018)",
             fontweight="bold", y=1.02, fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "envelope_liquidity.png", dpi=160, bbox_inches="tight")
plt.show()


## Solvency ratios — sector envelope

In [ ]:
solvency_ratios = ["debtRatio", "debtEquityRatio", "totalDebtToCapitalization"]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, ratio in zip(axes, solvency_ratios):
    pct = clean.groupby("year")[ratio].quantile(percentiles).unstack()
    ax.fill_between(pct.index, pct[0.10], pct[0.90],
                    alpha=0.20, color="#C00000", label="10th-90th pctile")
    ax.fill_between(pct.index, pct[0.25], pct[0.75],
                    alpha=0.40, color="#E07A1F", label="25th-75th pctile")
    ax.plot(pct.index, pct[0.50], marker="o", linewidth=2.5,
            color="#1F3864", label="Median")
    ax.set_title(f"{ratio}", fontweight="bold")
    ax.set_xlabel("Year"); ax.set_ylabel("Ratio")
    ax.set_xticks(pct.index)
    ax.legend(loc="best", fontsize=8)

plt.suptitle("SOLVENCY · Sector Volatility Envelope (2014-2018)",
             fontweight="bold", y=1.02, fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "envelope_solvency.png", dpi=160, bbox_inches="tight")
plt.show()


## Profitability ratios — sector envelope

In [ ]:
profitability_ratios = ["gross_margin", "op_margin", "net_margin"]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, ratio in zip(axes, profitability_ratios):
    pct = clean.groupby("year")[ratio].quantile(percentiles).unstack()
    ax.fill_between(pct.index, pct[0.10], pct[0.90],
                    alpha=0.20, color="#A9D08E", label="10th-90th pctile")
    ax.fill_between(pct.index, pct[0.25], pct[0.75],
                    alpha=0.40, color="#548235", label="25th-75th pctile")
    ax.plot(pct.index, pct[0.50], marker="o", linewidth=2.5,
            color="#1F3864", label="Median")
    ax.axhline(0, color="grey", linewidth=0.7, linestyle="--")
    ax.set_title(f"{ratio}", fontweight="bold")
    ax.set_xlabel("Year"); ax.set_ylabel("Margin")
    ax.set_xticks(pct.index)
    ax.legend(loc="best", fontsize=8)

plt.suptitle("PROFITABILITY · Sector Volatility Envelope (2014-2018)",
             fontweight="bold", y=1.02, fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "envelope_profitability.png", dpi=160, bbox_inches="tight")
plt.show()


## 5-year volatility profile (standard deviation of yearly medians)

In [ ]:
all_ratios = liquidity_ratios + solvency_ratios + profitability_ratios + ["returnOnAssets", "returnOnEquity"]
yearly_medians = clean.groupby("year")[all_ratios].median()

vol_profile = pd.DataFrame({
    "median_value":  yearly_medians.mean().round(3),
    "min_year":      yearly_medians.min().round(3),
    "max_year":      yearly_medians.max().round(3),
    "stdev_5yr":     yearly_medians.std().round(3),
    "coef_of_var":   (yearly_medians.std() / yearly_medians.mean().abs()).round(3),
})
vol_profile = vol_profile.sort_values("coef_of_var", ascending=False)

print("5-year volatility profile (sorted by coefficient of variation):")
print(vol_profile.to_string())

vol_profile.to_csv(OUTPUTS_DIR / "volatility_profile.csv")


## Cross-sectional dispersion summary table

In [ ]:
# For each ratio, show the 25/50/75/90 percentiles pooled across all 5 years
dispersion = pd.DataFrame({
    "10th_pctile": clean[all_ratios].quantile(0.10),
    "25th_pctile": clean[all_ratios].quantile(0.25),
    "median":      clean[all_ratios].quantile(0.50),
    "75th_pctile": clean[all_ratios].quantile(0.75),
    "90th_pctile": clean[all_ratios].quantile(0.90),
}).round(3)

print("Cross-sectional dispersion (pooled 2014-2018, all 1,200+ company-years):")
print(dispersion.to_string())

dispersion.to_csv(OUTPUTS_DIR / "cross_sectional_dispersion.csv")


✅ **Notebook 02 complete.** Move to `03_capital_budgeting_xlsx.ipynb`.